# 🏥 NeoLung AI — Notebook 1: Clinical Data & MLP Training
**Project:** NeoLung AI — Neonatal LUS RDS Triage | SRHU
**Notebook:** 1 of 3 — Clinical Data Pipeline

---
### ✅ FIRST TIME: Run All Cells (Runtime → Run All)
### 🔁 RETURNING AFTER TRAINING: Only run Cells 1, 2, 5 (define model), then use load snippet in Cell 8

---
What this notebook does:
1. Installs dependencies
2. Generates 500 synthetic clinical patients (3 severity classes)
3. Trains ClinicalMLP (8 inputs → 64 → 32 → 3 classes)
4. Evaluates on held-out test set
5. Saves checkpoint to HuggingFace Hub

In [ ]:
!pip install -q gradio scikit-learn pandas numpy torch huggingface_hub

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
import numpy as np
import pandas as pd

np.random.seed(42)
os.makedirs("data", exist_ok=True)
os.makedirs("models/clinical", exist_ok=True)

CLASS_PARAMS = {
    0: {"ga": (37, 1.5),  "bw": (2800, 400), "spo2": (97, 1.5),  "fio2": (0.21, 0.02),
        "rr": (45, 8),   "hr": (140, 15),  "cpap": (0, 0.5),    "temp": (36.8, 0.3)},
    1: {"ga": (32, 2),    "bw": (1800, 350), "spo2": (91, 3),     "fio2": (0.40, 0.1),
        "rr": (65, 10),  "hr": (155, 15),  "cpap": (5, 1),      "temp": (36.5, 0.4)},
    2: {"ga": (28, 2),    "bw": (1100, 300), "spo2": (84, 5),     "fio2": (0.65, 0.15),
        "rr": (80, 12),  "hr": (165, 15),  "cpap": (7, 1.5),    "temp": (36.2, 0.5)},
}
CLIP_RANGES = {
    "ga_weeks": (24, 42), "birth_weight_g": (500, 4000), "spo2_pct": (70, 100),
    "fio2": (0.21, 1.0), "rr_bpm": (20, 120), "hr_bpm": (80, 220),
    "cpap_cmh2o": (0, 12), "temperature_c": (34, 39),
}
COUNTS = {0: 200, 1: 150, 2: 150}
rows = []
for label, n in COUNTS.items():
    p = CLASS_PARAMS[label]
    for _ in range(n):
        row = {
            "ga_weeks":       np.random.normal(p["ga"][0],   p["ga"][1]),
            "birth_weight_g": np.random.normal(p["bw"][0],   p["bw"][1]),
            "spo2_pct":       np.random.normal(p["spo2"][0], p["spo2"][1]),
            "fio2":           np.random.normal(p["fio2"][0], p["fio2"][1]),
            "rr_bpm":         np.random.normal(p["rr"][0],   p["rr"][1]),
            "hr_bpm":         np.random.normal(p["hr"][0],   p["hr"][1]),
            "cpap_cmh2o":     np.random.normal(p["cpap"][0], p["cpap"][1]),
            "temperature_c":  np.random.normal(p["temp"][0], p["temp"][1]),
            "label": label,
        }
        for feat, (lo, hi) in CLIP_RANGES.items():
            row[feat] = float(np.clip(row[feat], lo, hi))
        rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("data/synthetic_clinical.csv", index=False)
print(f"Generated {len(df)} patients")
print(df["label"].value_counts().rename({0:"Normal", 1:"Moderate RDS", 2:"High RDS"}))
print("\nSaved to: data/synthetic_clinical.csv")

In [ ]:
import os, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)
os.makedirs("data", exist_ok=True)

df = pd.read_csv("data/synthetic_clinical.csv")
idx = list(range(len(df)))
labels = df["label"].tolist()

idx_train, idx_temp, _, y_temp = train_test_split(
    idx, labels, test_size=0.30, stratify=labels, random_state=42)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, stratify=y_temp, random_state=42)

splits = {"train": idx_train, "val": idx_val, "test": idx_test}
with open("data/clinical_splits.json", "w") as f:
    json.dump(splits, f)

label_names = {0: "Normal", 1: "Moderate RDS", 2: "High RDS"}
for split_name, split_idx in splits.items():
    sub = df.iloc[split_idx]
    counts = sub["label"].value_counts().sort_index()
    print(f"\n{split_name} ({len(sub)} patients):")
    for lbl, cnt in counts.items():
        print(f"  {label_names[lbl]}: {cnt}")
print("\nSaved to: data/clinical_splits.json")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset

torch.manual_seed(42)
np.random.seed(42)

FEATURE_COLS = [
    "ga_weeks", "birth_weight_g", "spo2_pct", "fio2",
    "rr_bpm", "hr_bpm", "cpap_cmh2o", "temperature_c"
]

class ClinicalMLP(nn.Module):
    """3-class severity classifier for neonatal RDS from 8 clinical features."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 3),
        )
    def forward(self, x):
        return self.net(x)

class ClinicalDataset(Dataset):
    """
    Loads clinical records by split index.
    Normalizes features using ONLY train-set min/max (no leakage).
    """
    def __init__(self, df, indices, feat_min=None, feat_max=None, fit=False):
        self.records = df.iloc[indices].reset_index(drop=True)
        feats = self.records[FEATURE_COLS].values.astype(np.float32)
        if fit:
            self.feat_min = feats.min(axis=0)
            self.feat_max = feats.max(axis=0)
        else:
            self.feat_min = feat_min
            self.feat_max = feat_max
        denom = np.where(self.feat_max - self.feat_min == 0, 1, self.feat_max - self.feat_min)
        self.features = (feats - self.feat_min) / denom
        self.labels = self.records["label"].values.astype(np.int64)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (
            torch.tensor(self.features[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx],   dtype=torch.long),
        )

print("ClinicalMLP and ClinicalDataset defined.")
print(f"Model parameters: {sum(p.numel() for p in ClinicalMLP().parameters()):,}")

In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter
from sklearn.metrics import f1_score

torch.manual_seed(42)
np.random.seed(42)
os.makedirs("models/clinical", exist_ok=True)

df = pd.read_csv("data/synthetic_clinical.csv")
with open("data/clinical_splits.json") as f:
    splits = json.load(f)

train_ds = ClinicalDataset(df, splits["train"], fit=True)
val_ds   = ClinicalDataset(df, splits["val"], feat_min=train_ds.feat_min, feat_max=train_ds.feat_max)

label_counts = Counter(train_ds.labels.tolist())
sampler = WeightedRandomSampler(
    [1.0 / label_counts[int(l)] for l in train_ds.labels],
    len(train_ds.labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

model = ClinicalMLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.5, 1.0, 1.5]).to(device))

EPOCHS, PATIENCE = 40, 8
best_val_f1, patience_counter = -1.0, 0

print(f"{'Epoch':>5} {'TrainLoss':>10} {'ValLoss':>10} {'ValMacroF1':>12}")
print("-" * 45)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(feats), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(feats)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for feats, labels in val_loader:
            feats, labels = feats.to(device), labels.to(device)
            logits = model(feats)
            val_loss += criterion(logits, labels).item() * len(feats)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    val_loss /= len(val_ds)
    val_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    scheduler.step()
    print(f"{epoch:>5} {train_loss:>10.4f} {val_loss:>10.4f} {val_f1:>12.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "best_val_macro_f1": best_val_f1,
                    "feat_min": train_ds.feat_min.tolist(),
                    "feat_max": train_ds.feat_max.tolist()},
                   "models/clinical/best_clinical.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nBest val Macro-F1: {best_val_f1:.4f}")
print("Checkpoint saved: models/clinical/best_clinical.pth")

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score

ckpt = torch.load("models/clinical/best_clinical.pth", map_location=device)
model = ClinicalMLP().to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

df = pd.read_csv("data/synthetic_clinical.csv")
with open("data/clinical_splits.json") as f:
    splits = json.load(f)
test_ds = ClinicalDataset(df, splits["test"],
    feat_min=np.array(ckpt["feat_min"]), feat_max=np.array(ckpt["feat_max"]))
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for feats, labels in test_loader:
        all_preds.extend(model(feats.to(device)).argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.tolist())

target_names = ["Normal", "Moderate RDS", "High RDS"]
print("=" * 55)
print("TEST SET EVALUATION")
print("=" * 55)
print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))
print(f"Macro F1:    {f1_score(all_labels, all_preds, average='macro',    zero_division=0):.4f}")
print(f"Weighted F1: {f1_score(all_labels, all_preds, average='weighted', zero_division=0):.4f}")

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix (rows=True, cols=Pred):")
print(f"{'':15}" + "".join(f"{n:>14}" for n in target_names))
for i, row in enumerate(cm):
    print(f"{target_names[i]:15}" + "".join(f"{v:>14}" for v in row))

In [ ]:
# --- HUGGINGFACE CHECKPOINT SAVE ---
# Setup: In Colab left panel → key icon (Secrets) → Add: Name=HF_TOKEN, Value=your_token

import json
import os
from huggingface_hub import HfApi, login
from google.colab import userdata

token = userdata.get('HF_TOKEN')
login(token=token)

api = HfApi()
repo_id = "YOUR_HF_USERNAME/NeoLung-clinical"  # <-- CHANGE THIS to your HF username

api.create_repo(repo_id=repo_id, exist_ok=True)

api.upload_file(
    path_or_fileobj="models/clinical/best_clinical.pth",
    path_in_repo="best_clinical.pth",
    repo_id=repo_id
)

config = {
    "input_dim": 8,
    "hidden_dims": [64, 32],
    "output_dim": 3,
    "classes": ["Normal", "Moderate RDS", "High RDS"],
    "features": ["ga_weeks","birth_weight_g","spo2_pct","fio2","rr_bpm","hr_bpm","cpap_cmh2o","temperature_c"]
}
with open("models/clinical/model_config.json","w") as f:
    json.dump(config, f, indent=2)

api.upload_file(
    path_or_fileobj="models/clinical/model_config.json",
    path_in_repo="model_config.json",
    repo_id=repo_id
)
print(f"✅ Uploaded to: https://huggingface.co/{repo_id}")

## 🔁 RETURNING TO THIS NOTEBOOK AFTER TRAINING?

Run ONLY these cells:
1. Cell 1 — Install (Colab resets packages on every session)
2. Cell 2 — GPU check
3. Cell 5 — Define ClinicalMLP class (needed before loading weights)
4. Cell 8 — HF login

Then load weights with this snippet:
```python
from huggingface_hub import hf_hub_download
path = hf_hub_download(repo_id="YOUR_HF_USERNAME/NeoLung-clinical", filename="best_clinical.pth")
model = ClinicalMLP().to(device)
model.load_state_dict(torch.load(path, map_location=device))
model.eval()
print("✅ Weights loaded from HuggingFace")
```

SKIP: Cells 3, 4, 6, 7 — data generation and training already done

In [ ]:
# Backup: Download .pth file directly to your PC
from google.colab import files
files.download("models/clinical/best_clinical.pth")